In [1]:
import plotly.express as px
import pandas as pd

MasterDataFrame = pd.read_csv('CleanedMasterDataFrame.csv')
DropColumn = list(MasterDataFrame.columns)
MasterDataFrame = MasterDataFrame.drop(columns=DropColumn[0])

IsoToCountryDataFrame = pd.read_excel('CLASS_2025_10_07.xlsx')
ColumnsToKeep = ['Economy','Code']
DropColumn = [i for i in IsoToCountryDataFrame.columns if i not in ColumnsToKeep]
IsoToCountryDataFrame = IsoToCountryDataFrame.drop(columns=DropColumn)
IsoToCountryDataFrame = IsoToCountryDataFrame.rename(columns={'Economy':'country','Code':'countryiso3code'})

In [2]:
SplitByCountry = [group for _,group in MasterDataFrame.groupby('countryiso3code')] #splitting based on country
                                                                                   #should return a list of dataframes
CountryAndMeanGiniValueDataFrame = pd.DataFrame({'countryiso3code':[],
                                                 'AverageGiniValue':[]})

for iterator in SplitByCountry: #iterator should be a dataframe
    CountryCode = iterator['countryiso3code'].iloc[0]
    MeanGiniValue = iterator['SI.POV.GINI'].mean(skipna=True)
    CountryAndMeanGiniValueDataFrame.loc[len(CountryAndMeanGiniValueDataFrame)] = [CountryCode,MeanGiniValue]

CountryAndMeanGiniValueDataFrame = pd.merge(CountryAndMeanGiniValueDataFrame,IsoToCountryDataFrame,
                                            how='left',on='countryiso3code')

MaxAverageGiniValue = CountryAndMeanGiniValueDataFrame['AverageGiniValue'].max()
MinAverageGiniValue = CountryAndMeanGiniValueDataFrame['AverageGiniValue'].min()

In [ ]:
ChoroplethMap = px.choropleth(
    CountryAndMeanGiniValueDataFrame,
    locations=CountryAndMeanGiniValueDataFrame['countryiso3code'],
    locationmode='ISO-3',
    scope='world',
    hover_name=CountryAndMeanGiniValueDataFrame['country'],
    hover_data={'AverageGiniValue':':.2f','countryiso3code':False},
    title='Average Gini Value from 1990-2020 around the world',
    color='AverageGiniValue',
    color_continuous_scale='Viridis',
    range_color=[MinAverageGiniValue,MaxAverageGiniValue]
)

ChoroplethMap.update_geos(visible=True,
    projection_scale=1,  
    center=dict(lat=0, lon=0),  
    lataxis_range=[-60, 90],
    lonaxis_range=[-180, 180],
)

ChoroplethMap.update_layout(
    title_font_family = 'Sans-Serif',
    coloraxis_colorbar_title = 'Average Gini Value',
    coloraxis_colorbar_title_font_family = 'Sans-Serif',
    dragmode=False,
    margin={'r':0,'t':40,'l':0,'b':0},
)

ChoroplethMap.update_layout(
    title = {
        'x' : 0.5,
        'y' : 0.95,
        'xanchor' : 'center',
        'yanchor' : 'top'
    },
)

ChoroplethMap.show(config={'scrollZoom':False,'displayModeBar':False})